In [2]:
import pandas as pd

In [3]:
df=pd.read_excel('C:/Users/Ahmed Taher/Desktop/DEPI ASSIGNMENTS/GRADUATION PROJECT/Datasets/Datasets/Manufacturing Downtime/Manufacturing_Line_Productivity.xlsx')
df
sheets=pd.ExcelFile('C:/Users/Ahmed Taher/Desktop/DEPI ASSIGNMENTS/GRADUATION PROJECT/Datasets/Datasets/Manufacturing Downtime/Manufacturing_Line_Productivity.xlsx').sheet_names
sheets

['Line productivity', 'Products', 'Downtime factors', 'Line downtime']

In [4]:
Line_productivity= pd.read_excel('C:/Users/Ahmed Taher/Desktop/DEPI ASSIGNMENTS/GRADUATION PROJECT/Datasets/Datasets/Manufacturing Downtime/Manufacturing_Line_Productivity.xlsx',sheet_name='Line productivity')
Products= pd.read_excel('C:/Users/Ahmed Taher/Desktop/DEPI ASSIGNMENTS/GRADUATION PROJECT/Datasets/Datasets/Manufacturing Downtime/Manufacturing_Line_Productivity.xlsx',sheet_name='Products')
Downtime_factors= pd.read_excel('C:/Users/Ahmed Taher/Desktop/DEPI ASSIGNMENTS/GRADUATION PROJECT/Datasets/Datasets/Manufacturing Downtime/Manufacturing_Line_Productivity.xlsx',sheet_name='Downtime factors')
Line_downtime= pd.read_excel('C:/Users/Ahmed Taher/Desktop/DEPI ASSIGNMENTS/GRADUATION PROJECT/Datasets/Datasets/Manufacturing Downtime/Manufacturing_Line_Productivity.xlsx',sheet_name='Line downtime',header=1)

In [5]:
Line_productivity

,Date,Product,Batch,Operator,Start Time,End Time
0,2024-08-29,OR-600,422111,Mac,11:50:00,14:05:00
1,2024-08-29,LE-600,422112,Mac,14:05:00,15:45:00
2,2024-08-29,LE-600,422113,Mac,15:45:00,17:35:00
3,2024-08-29,LE-600,422114,Mac,17:35:00,19:15:00
4,2024-08-29,LE-600,422115,Charlie,19:15:00,20:39:00
5,2024-08-29,LE-600,422116,Charlie,20:39:00,21:39:00
6,2024-08-29,LE-600,422117,Charlie,21:39:00,22:54:00
7,2024-08-30,CO-600,422118,Dee,04:05:00,06:05:00
8,2024-08-30,CO-600,422119,Dee,06:05:00,07:30:00
9,2024-08-30,CO-600,422120,Dee,07:30:00,09:22:00


In [6]:
Line_productivity.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 38 entries, 0 to 37
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   Date        38 non-null     datetime64[ns]
 1   Product     38 non-null     object        
 2   Batch       38 non-null     int64         
 3   Operator    38 non-null     object        
 4   Start Time  38 non-null     object        
 5   End Time    38 non-null     object        
dtypes: datetime64[ns](1), int64(1), object(4)
memory usage: 1.9+ KB


In [7]:
Line_productivity.shape

(38, 6)

In [8]:
Line_productivity["Start Time"] = pd.to_datetime(Line_productivity["Start Time"], format="%H:%M:%S", errors="coerce")
Line_productivity["End Time"] = pd.to_datetime(Line_productivity["End Time"], format="%H:%M:%S", errors="coerce")

In [9]:
Line_productivity.dtypes

Date          datetime64[ns]
Product               object
Batch                  int64
Operator              object
Start Time    datetime64[ns]
End Time      datetime64[ns]
dtype: object

In [10]:
Line_productivity.duplicated().sum()

np.int64(0)

In [11]:
Line_productivity.isnull().sum()

Date          0
Product       0
Batch         0
Operator      0
Start Time    0
End Time      0
dtype: int64

In [12]:
start = pd.to_datetime(Line_productivity["Start Time"], format="%H:%M:%S")
end   = pd.to_datetime(Line_productivity["End Time"], format="%H:%M:%S")
end = end.where(end >= start, end + pd.Timedelta(days=1))
Line_productivity["Actual time"] = (end - start).dt.total_seconds() / 60

In [13]:
Line_productivity["Shift"] = Line_productivity["Start Time"].apply(
    lambda t: "Morning" if 6 <= t.hour < 14
            else "Afternoon" if 14 <= t.hour < 22
            else "Night")

In [14]:
Line_productivity['Actual time'] = Line_productivity['Actual time'].astype(int)

In [15]:
Line_productivity['Start Time'] = Line_productivity['Start Time'].dt.time
Line_productivity['End Time']   = Line_productivity['End Time'].dt.time

In [16]:
Products

,Product,Flavor,Size,Min batch time
0,OR-600,Orange,600 ml,60
1,LE-600,Lemon lime,600 ml,60
2,CO-600,Cola,600 ml,60
3,DC-600,Diet Cola,600 ml,60
4,RB-600,Root Berry,600 ml,60
5,CO-2L,Cola,2 L,98


In [17]:
Products["Size"] = (
    Products["Size"]
    .str.replace("ml", "", regex=False)     
    .str.replace("L", "000", regex=False)  
    .str.replace(" ", "", regex=False)     
    .astype(int)                          
)

In [18]:
Products.rename(columns={'Size':'Size(ml)'},inplace=True)
Products

,Product,Flavor,Size(ml),Min batch time
0,OR-600,Orange,600,60
1,LE-600,Lemon lime,600,60
2,CO-600,Cola,600,60
3,DC-600,Diet Cola,600,60
4,RB-600,Root Berry,600,60
5,CO-2L,Cola,2000,98


In [19]:
Products.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 4 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   Product         6 non-null      object
 1   Flavor          6 non-null      object
 2   Size(ml)        6 non-null      int64 
 3   Min batch time  6 non-null      int64 
dtypes: int64(2), object(2)
memory usage: 324.0+ bytes


In [20]:
Products.duplicated().sum()

np.int64(0)

In [21]:
Products.shape

(6, 4)

In [22]:
mask_cola_600 = (Products["Flavor"] == "Cola") & (Products["Size(ml)"] == 600)
mask_cola_2L  = (Products["Flavor"] == "Cola") & (Products["Size(ml)"] == 2000)

Products.loc[mask_cola_600, "Product"] = "Cola (600 ml)"
Products.loc[mask_cola_2L,  "Product"] = "Cola (2L)"

In [23]:
Products.rename(
    columns={"Min batch time": "Min batch time (mins)"},
    inplace=True
)

In [24]:
Products.isnull().sum()

Product                  0
Flavor                   0
Size(ml)                 0
Min batch time (mins)    0
dtype: int64

In [25]:
Downtime_factors

,Factor,Description,Operator Error
0,1,Emergency stop,No
1,2,Batch change,Yes
2,3,Labeling error,No
3,4,Inventory shortage,No
4,5,Product spill,Yes
5,6,Machine adjustment,Yes
6,7,Machine failure,No
7,8,Batch coding error,Yes
8,9,Conveyor belt jam,No
9,10,Calibration error,Yes


In [26]:
Downtime_factors.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12 entries, 0 to 11
Data columns (total 3 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   Factor          12 non-null     int64 
 1   Description     12 non-null     object
 2   Operator Error  12 non-null     object
dtypes: int64(1), object(2)
memory usage: 420.0+ bytes


In [27]:
Downtime_factors.shape

(12, 3)

In [28]:
Downtime_factors.duplicated().sum()

np.int64(0)

In [29]:
Downtime_factors.isnull().sum()

Factor            0
Description       0
Operator Error    0
dtype: int64

In [30]:
Line_downtime

,Batch,1,2,3,4,5,6,7,8,9,10,11,12
0,422111,NaN,60.0,NaN,NaN,NaN,NaN,15.0,NaN,NaN,NaN,NaN,NaN
1,422112,NaN,20.0,NaN,NaN,NaN,NaN,NaN,20.0,NaN,NaN,NaN,NaN
2,422113,NaN,50.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,422114,NaN,NaN,NaN,25.0,NaN,15.0,NaN,NaN,NaN,NaN,NaN,NaN
4,422115,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,24.0,NaN,NaN
5,422116,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,422117,NaN,10.0,NaN,NaN,NaN,5.0,NaN,NaN,NaN,NaN,NaN,NaN
7,422118,NaN,NaN,NaN,NaN,NaN,14.0,16.0,NaN,NaN,NaN,10.0,20.0
8,422119,NaN,NaN,NaN,25.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,422120,NaN,NaN,NaN,20.0,15.0,NaN,NaN,NaN,17.0,NaN,NaN,NaN


In [31]:
Line_downtime.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 38 entries, 0 to 37
Data columns (total 13 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Batch   38 non-null     int64  
 1   1       0 non-null      float64
 2   2       5 non-null      float64
 3   3       2 non-null      float64
 4   4       9 non-null      float64
 5   5       3 non-null      float64
 6   6       12 non-null     float64
 7   7       11 non-null     float64
 8   8       6 non-null      float64
 9   9       1 non-null      float64
 10  10      3 non-null      float64
 11  11      3 non-null      float64
 12  12      6 non-null      float64
dtypes: float64(12), int64(1)
memory usage: 4.0 KB


In [32]:
Line_downtime.shape

(38, 13)

In [33]:
Line_downtime_long = (
    Line_downtime.melt(
        id_vars=["Batch"],           
        value_vars=list(range(1, 13)),
        var_name="Factor",           
        value_name="Downtime_Minutes" 
    )
    .dropna(subset=["Downtime_Minutes"]) 
    .sort_values(["Batch", "Factor"])   
        
    .reset_index(drop=True)
)

In [34]:
Line_downtime_long

,Batch,Factor,Downtime_Minutes
0,422111,2,60.0
1,422111,7,15.0
2,422112,2,20.0
3,422112,8,20.0
4,422113,2,50.0
...,...,...,...
56,422147,4,17.0
57,422147,6,60.0
58,422147,7,30.0
59,422148,4,25.0


In [35]:
Line_downtime_long['Downtime_Minutes'] =Line_downtime_long['Downtime_Minutes'].astype(int)

In [36]:
Line_downtime_long.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 61 entries, 0 to 60
Data columns (total 3 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   Batch             61 non-null     int64 
 1   Factor            61 non-null     object
 2   Downtime_Minutes  61 non-null     int64 
dtypes: int64(2), object(1)
memory usage: 1.6+ KB


In [37]:
Line_downtime_long.duplicated().sum()

np.int64(0)

In [38]:
Line_downtime_long.isnull().sum()

Batch               0
Factor              0
Downtime_Minutes    0
dtype: int64

In [39]:
#Analysis

In [40]:
# What is the total downtime recorded?
total_downtime=Line_downtime_long['Downtime_Minutes'].sum()
total_downtime

np.int64(1388)

In [41]:
# How many downtime events occurred?
count_downtime=Line_downtime_long['Downtime_Minutes'].count()
count_downtime

np.int64(61)

In [42]:
#What is the total number of batches?
count_Batches=Line_productivity['Batch'].count()
count_Batches

np.int64(38)

In [43]:
#What is the total scheduled production time? (batches time)
total_scheduled_time = Line_productivity['Actual time'].sum()
total_scheduled_time

np.int64(3858)

In [44]:
# How much time was actually spent producing? 
total_Actualtime = Line_productivity['Actual time'].sum()      
total_downtime=Line_downtime_long['Downtime_Minutes'].sum()
total_productive_time = total_Actualtime- total_downtime
total_productive_time

np.int64(2470)

In [45]:
#what is the percentage of lost production?
Downtime_impact_percent = (Line_downtime_long['Downtime_Minutes'].sum()/ Line_productivity['Actual time'].sum()) * 100
Downtime_impact_percent

np.float64(35.97719025401763)

In [46]:
#Does downtime intensity correlate with total batches produced per day?

import seaborn as sns

corr_value =Line_downtime_long['Downtime_Minutes'].corr(Line_downtime_long["Batch"])
corr_value

np.float64(0.06219723287427554)

In [47]:
#How does daily production volume change over time?
daily_production = ( Line_productivity.groupby('Date')['Batch'].count()
                 .reset_index()
                 .sort_values('Date'))
daily_production 

,Date,Batch
0,2024-08-29,7
1,2024-08-30,12
2,2024-08-31,7
3,2024-09-02,11
4,2024-09-03,1


In [48]:
#What is the min batch time, downtime, and total duration for each batch?
merged = pd.merge(Line_downtime_long, Line_productivity, on="Batch", how="inner")
merged2 = pd.merge(merged,Products, on="Product",how="inner")
batch_summary = merged2.groupby('Batch').agg(
    Min_Batch_Time=('Min batch time', 'min'),
    Total_Downtime=('Downtime_Minutes', 'sum'),
    Total_Duration=('Actual time', 'first')
).reset_index()
batch_summary

KeyError: "Column(s) ['Min batch time'] do not exist"

In [ ]:
#What is the average batch duration per product and operator?
avg_batch_duration = (
    Line_productivity.groupby(['Product', 'Operator'])['Actual time'].mean()
    .reset_index()
    .rename(columns={'Actual time': 'Avg_Batch_Duration_Minute'})
    .sort_values('Avg_Batch_Duration_Minute', ascending=False))
avg_batch_duration 

In [69]:
#Which operator has the highest downtime?
batch_downtime = (
    Line_downtime_long.groupby('Batch', as_index=False)
    .agg({'Downtime_Minutes': 'sum'})
)
merged = pd.merge(Line_productivity, batch_downtime, on="Batch", how="left")

downtime_by_operator = ( merged.groupby('Operator')['Downtime_Minutes'].sum()
             .sort_values(ascending=False))
downtime_by_operator


Operator
Charlie    384.0
Dee        370.0
Mac        332.0
Dennis     302.0
Name: Downtime_Minutes, dtype: float64

In [70]:
#Which product–flavor–size combination shows the highest deviation from ideal performance?

Products["Deviation"] = abs(Line_productivity["Actual time"] - Products["Min batch time"])


product_flavor= (
    Products.groupby(["Product", "Flavor", "Size(ml)"], as_index=False)["Deviation"]
    .mean()
    .sort_values(by="Deviation", ascending=False)
    .head(1)
)

product_flavor

KeyError: 'Min batch time'

In [71]:
#Which products have the highest number of batches produced?
batches_per_product = (Line_productivity.groupby('Product')['Batch'].nunique()
                     .reset_index(name='Number_of_Batches')
                     .sort_values('Number_of_Batches', ascending=False))
batches_per_product

,Product,Number_of_Batches
1,CO-600,15
5,RB-600,7
3,LE-600,6
0,CO-2L,5
2,DC-600,4
4,OR-600,1


In [72]:
# Which products have the most downtime?
downtime_by_product = (merged.groupby('Product')['Downtime_Minutes'].sum()
                      .sort_values(ascending=False))
downtime_by_product

Product
CO-600    494.0
CO-2L     277.0
RB-600    258.0
LE-600    169.0
DC-600    115.0
OR-600     75.0
Name: Downtime_Minutes, dtype: float64

In [68]:
#Which products consume the most production time?
production_time_by_product = (Line_productivity.groupby('Product')['Actual time'].sum()
                             .reset_index()
                             .sort_values('Actual time', ascending=False))
production_time_by_product

,Product,Actual time
1,CO-600,1394
0,CO-2L,767
5,RB-600,678
3,LE-600,529
2,DC-600,355
4,OR-600,135


In [73]:
#What is the downtime % per product?
batch_downtime = (
    Line_downtime_long.groupby('Batch', as_index=False)
    .agg({'Downtime_Minutes': 'sum'})
)

merged = pd.merge(Line_productivity, batch_downtime, on='Batch', how='left')

downtime_impact = (
    merged.groupby('Product', as_index=False)
    .agg({
        'Downtime_Minutes': 'sum',
        'Actual time': 'sum'
    })
)

downtime_impact['Downtime Impact %'] = (
    downtime_impact['Downtime_Minutes'] / downtime_impact['Actual time'] * 100
)


downtime_impact = downtime_impact.sort_values('Downtime Impact %', ascending=False)


In [74]:
downtime_impact

,Product,Downtime_Minutes,Actual time,Downtime Impact %
4,OR-600,75.0,135,55.555556
5,RB-600,258.0,678,38.053097
0,CO-2L,277.0,767,36.114733
1,CO-600,494.0,1394,35.437590
2,DC-600,115.0,355,32.394366
3,LE-600,169.0,529,31.947070


In [49]:
merged

,Batch,Factor,Downtime_Minutes,Date,Product,Operator,Start Time,End Time,Actual time,Shift
0,422111,2,60,2024-08-29,OR-600,Mac,11:50:00,14:05:00,135,Morning
1,422111,7,15,2024-08-29,OR-600,Mac,11:50:00,14:05:00,135,Morning
2,422112,2,20,2024-08-29,LE-600,Mac,14:05:00,15:45:00,100,Afternoon
3,422112,8,20,2024-08-29,LE-600,Mac,14:05:00,15:45:00,100,Afternoon
4,422113,2,50,2024-08-29,LE-600,Mac,15:45:00,17:35:00,110,Afternoon
...,...,...,...,...,...,...,...,...,...,...
56,422147,4,17,2024-09-02,CO-2L,Charlie,19:30:00,22:55:00,205,Afternoon
57,422147,6,60,2024-09-02,CO-2L,Charlie,19:30:00,22:55:00,205,Afternoon
58,422147,7,30,2024-09-02,CO-2L,Charlie,19:30:00,22:55:00,205,Afternoon
59,422148,4,25,2024-09-03,CO-2L,Mac,22:55:00,01:05:00,130,Night


In [50]:
# Which operator has the highest downtime
operator_downtime = (
    merged.groupby('Operator', as_index=False)
          .agg({'Downtime_Minutes': 'sum'})
)

operator_downtime = operator_downtime.sort_values('Downtime_Minutes', ascending=False)
top_operator = operator_downtime.iloc[0]
top_operator


Operator            Charlie
Downtime_Minutes        384
Name: 0, dtype: object

In [51]:
#How many batches does each operator handle?
batches_per_operator = (
   Line_productivity.groupby('Operator')['Batch'].nunique() 
                    .reset_index(name='Number_of_Batches')
                    .sort_values('Number_of_Batches', ascending=False))

batches_per_operator

,Operator,Number_of_Batches
0,Charlie,11
1,Dee,11
2,Dennis,8
3,Mac,8


In [52]:
#How does each operator’s efficiency change with their downtime?
efficiency_by_operator_shift = (
    merged.groupby(['Operator', 'Shift'])
    .agg({'Actual time': 'sum',
          'Downtime_Minutes': 'sum'})
    .reset_index()
)
efficiency_by_operator_shift['Operator Efficiency (%)'] = (
    (efficiency_by_operator_shift['Actual time'] - efficiency_by_operator_shift['Downtime_Minutes']) /
    efficiency_by_operator_shift['Actual time'] * 100
)
efficiency_by_operator_shift['Operator Efficiency (%)'] = (
    efficiency_by_operator_shift['Operator Efficiency (%)']
    .round(0)
)
efficiency_by_operator_shift = efficiency_by_operator_shift[['Operator', 'Shift', 'Downtime_Minutes', 'Operator Efficiency (%)']]

efficiency_by_operator_shift 

,Operator,Shift,Downtime_Minutes,Operator Efficiency (%)
0,Charlie,Afternoon,369,82.0
1,Charlie,Night,15,80.0
2,Dee,Morning,147,81.0
3,Dee,Night,223,82.0
4,Dennis,Afternoon,40,80.0
5,Dennis,Morning,262,77.0
6,Mac,Afternoon,175,76.0
7,Mac,Morning,125,74.0
8,Mac,Night,32,88.0


In [67]:
# How does each operator’s efficiency change with their downtime?
efficiency_by_operator = (
    merged.groupby(['Operator'])
    .agg({
        'Actual time': 'sum',
        'Downtime_Minutes': 'sum'
    })
    .reset_index()
)
efficiency_by_operator['Operator Efficiency (%)'] = (
    (efficiency_by_operator['Actual time'] - efficiency_by_operator['Downtime_Minutes']) /
    efficiency_by_operator['Actual time'] * 100
).round(0)
efficiency_by_operator = efficiency_by_operator[
    ['Operator', 'Downtime_Minutes', 'Operator Efficiency (%)']
]
efficiency_by_operator

,Operator,Downtime_Minutes,Operator Efficiency (%)
0,Charlie,384,82.0
1,Dee,370,81.0
2,Dennis,302,77.0
3,Mac,332,77.0


In [66]:
efficiency_by_operator_shift = (
    merged.groupby(['Operator', 'Shift'])
    .agg({
        'Actual time': 'sum',
        'Downtime_Minutes': 'sum',
        'Batch': 'count'          # ← NEW: number of batches
    })
    .reset_index()
)

# Calculate efficiency
efficiency_by_operator_shift['Operator Efficiency (%)'] = (
    (efficiency_by_operator_shift['Actual time'] - efficiency_by_operator_shift['Downtime_Minutes']) /
    efficiency_by_operator_shift['Actual time'] * 100
).round(0)

# Reorder columns
efficiency_by_operator_shift = efficiency_by_operator_shift[
    ['Operator', 'Shift', 'Batch', 'Downtime_Minutes', 'Operator Efficiency (%)']
]

efficiency_by_operator_shift


,Operator,Shift,Batch,Downtime_Minutes,Operator Efficiency (%)
0,Charlie,Afternoon,16,369,82.0
1,Charlie,Night,1,15,80.0
2,Dee,Morning,8,147,81.0
3,Dee,Night,11,223,82.0
4,Dennis,Afternoon,2,40,80.0
5,Dennis,Morning,10,262,77.0
6,Mac,Afternoon,7,175,76.0
7,Mac,Morning,4,125,74.0
8,Mac,Night,2,32,88.0


In [53]:
#Is most of the downtime caused by operators or by system issues?
merged2 = pd.merge(Line_downtime_long, Downtime_factors, on="Factor", how="inner")
downtime_by_source = (merged2.groupby('Operator Error')['Downtime_Minutes'].sum()
                    .reset_index()
                    .sort_values('Downtime_Minutes', ascending=False))
downtime_by_source['Percentage'] = (downtime_by_source['Downtime_Minutes'] / downtime_by_source['Downtime_Minutes'].sum() * 100)
downtime_by_source['Percentage']
downtime_by_source['Operator Error'] = downtime_by_source['Operator Error'].replace({'Yes': 'Operator','No': 'System'})
downtime_by_source


,Operator Error,Downtime_Minutes,Percentage
1,Operator,776,55.907781
0,System,612,44.092219


In [54]:
#What is the total number of downtime factors?
Downtime_factors['Description'].count()

np.int64(12)

In [63]:
# What is the most time consuming by downtime factor?
factor_downtime = (
 merged2.groupby('Description', as_index=False)
    .agg({'Downtime_Minutes': 'sum'})
)

factor_downtime = factor_downtime.sort_values('Downtime_Minutes', ascending=False)
factor_downtime

,Description,Downtime_Minutes
7,Machine adjustment,332
8,Machine failure,254
4,Inventory shortage,225
0,Batch change,160
1,Batch coding error,145
9,Other,74
10,Product spill,57
2,Calibration error,49
6,Labeling error,42
5,Label switch,33


In [65]:
# What is the most occurred downtime factor?

factor_counts = (
  merged2.groupby('Description', as_index=False)
    .size()
    .rename(columns={'size': 'Count'})
)
most_occurred_factor = factor_counts.sort_values('Count', ascending=False)
most_occurred_factor

,Description,Count
7,Machine adjustment,12
8,Machine failure,11
4,Inventory shortage,9
1,Batch coding error,6
9,Other,6
0,Batch change,5
2,Calibration error,3
5,Label switch,3
10,Product spill,3
6,Labeling error,2


In [57]:
#How much cumulative downtime did each factor contribute over time?
downtime_by_factor = (
    merged2.groupby('Description', as_index=False)['Downtime_Minutes']
    .sum()
    .sort_values('Downtime_Minutes', ascending=False)
)

total_downtime = downtime_by_factor['Downtime_Minutes'].sum()
downtime_by_factor['Downtime_%'] = (downtime_by_factor['Downtime_Minutes'] / total_downtime) * 100

downtime_by_factor['Cumulative_%'] = downtime_by_factor['Downtime_%'].cumsum()

downtime_by_factor

,Description,Downtime_Minutes,Downtime_%,Cumulative_%
7,Machine adjustment,332,23.919308,23.919308
8,Machine failure,254,18.299712,42.219020
4,Inventory shortage,225,16.210375,58.429395
0,Batch change,160,11.527378,69.956772
1,Batch coding error,145,10.446686,80.403458
9,Other,74,5.331412,85.734870
10,Product spill,57,4.106628,89.841499
2,Calibration error,49,3.530259,93.371758
6,Labeling error,42,3.025937,96.397695
5,Label switch,33,2.377522,98.775216


In [58]:
#Are certain products more affected by specific downtime factors?
merged_final = (
    Line_downtime_long
    .merge(Line_productivity[['Batch', 'Product', 'Date', 'Shift']], on='Batch', how='inner')
    .merge(Downtime_factors[['Factor', 'Description']], on='Factor', how='left')
)

product_factor_impact = (
    merged_final.groupby(['Product', 'Description'], as_index=False)['Downtime_Minutes']
    .sum()
    .sort_values(['Product', 'Downtime_Minutes'], ascending=[True, False])
)

product_factor_impact['Total_Product_Downtime'] = (
    product_factor_impact.groupby('Product')['Downtime_Minutes'].transform('sum')
)
product_factor_impact['Downtime_%'] = (
    product_factor_impact['Downtime_Minutes'] / product_factor_impact['Total_Product_Downtime'] * 100
)

product_factor_impact


,Product,Description,Downtime_Minutes,Total_Product_Downtime,Downtime_%
3,CO-2L,Machine adjustment,120,277,43.321300
4,CO-2L,Machine failure,55,277,19.855596
1,CO-2L,Inventory shortage,42,277,15.162455
0,CO-2L,Batch coding error,31,277,11.191336
2,CO-2L,Labeling error,22,277,7.942238
5,CO-2L,Other,7,277,2.527076
13,CO-600,Machine failure,116,494,23.481781
10,CO-600,Inventory shortage,108,494,21.862348
12,CO-600,Machine adjustment,57,494,11.538462
15,CO-600,Product spill,57,494,11.538462


In [59]:
# Which downtime factors affect each product the most?
product_factor = (
    merged_final.groupby(['Product', 'Description'], as_index=False)['Downtime_Minutes']
    .sum()
)

product_factor['Rank'] = (
    product_factor.groupby('Product')['Downtime_Minutes'].rank(method='first', ascending=False)
)

top_factor_per_product = product_factor[product_factor['Rank'] == 1].drop(columns='Rank')

top_factor_per_product = top_factor_per_product.sort_values('Downtime_Minutes', ascending=False)

top_factor_per_product

,Product,Description,Downtime_Minutes
32,RB-600,Machine adjustment,135
3,CO-2L,Machine adjustment,120
13,CO-600,Machine failure,116
20,LE-600,Batch change,80
25,OR-600,Batch change,60
18,DC-600,Machine failure,50


In [60]:
merged3 = (
    Line_downtime_long
    .merge(Line_productivity[['Batch', 'Operator', 'Date']], 
           on='Batch', how='inner')
    .merge(Downtime_factors[['Factor', 'Description', 'Operator Error']], 
           on='Factor', how='left')
)

operator_related = merged3[
    (merged3['Operator Error'] == True) | 
    (merged3['Operator Error'].astype(str).str.lower().isin(['yes', 'true']))
]

operator_downtime = (
    operator_related.groupby('Operator', as_index=False)['Downtime_Minutes']
    .sum()
    .sort_values('Downtime_Minutes', ascending=False)
)

operator_downtime

,Operator,Downtime_Minutes
0,Charlie,228
1,Dee,192
3,Mac,192
2,Dennis,164


In [61]:
#Which downtime factors occur most often in each shift?
merged4 = (
    Line_downtime_long
    .merge(Line_productivity[['Batch', 'Operator', 'Date', 'Shift']], 
           on='Batch', how='inner')
    .merge(Downtime_factors[['Factor', 'Description', 'Operator Error']], 
           on='Factor', how='left')
)

downtime_by_shift_factor = (merged4.groupby(['Date','Shift', 'Description']).size()
                         .reset_index(name='Count')
                         .sort_values(['Shift', 'Count'], ascending=[True, False]))
downtime_by_shift_factor

,Date,Shift,Description,Count
0,2024-08-29,Afternoon,Batch change,3
4,2024-08-29,Afternoon,Machine adjustment,2
9,2024-08-30,Afternoon,Machine adjustment,2
12,2024-08-30,Afternoon,Product spill,2
30,2024-09-02,Afternoon,Machine adjustment,2
31,2024-09-02,Afternoon,Machine failure,2
1,2024-08-29,Afternoon,Batch coding error,1
2,2024-08-29,Afternoon,Calibration error,1
3,2024-08-29,Afternoon,Inventory shortage,1
7,2024-08-30,Afternoon,Batch coding error,1


In [62]:
# Which downtime factors occur most often in each day?
downtime_by_day_factor = (merged4.groupby(['Date', 'Description']).size()
                         .reset_index(name='Count')
                         .sort_values(['Count'], ascending=[ False]))
downtime_by_day_factor

,Date,Description,Count
25,2024-09-02,Machine adjustment,7
11,2024-08-30,Machine failure,5
0,2024-08-29,Batch change,4
12,2024-08-30,Other,3
8,2024-08-30,Inventory shortage,3
10,2024-08-30,Machine adjustment,3
26,2024-09-02,Machine failure,3
13,2024-08-30,Product spill,3
4,2024-08-29,Machine adjustment,2
9,2024-08-30,Label switch,2
